In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import rcParams
from ipywidgets import IntSlider, interact

rcParams['font.family'] = 'SimHei'
rcParams['axes.unicode_minus'] = False

In [6]:
df = pd.read_csv("../data/raw/may_filtered.csv")

In [7]:
df.head()

,date,time,duration,red,ied,accX,accY,accZ
0,02-05-2026,16:21:44,0:00:00,14936049.0,14798220.0,1100.0,-791.0,1403.0
1,02-05-2026,16:21:44,0:00:00,14938572.0,14801393.0,1100.0,-791.0,1403.0
2,02-05-2026,16:21:44,0:00:00,14941545.0,14804653.0,1100.0,-791.0,1403.0
3,02-05-2026,16:21:44,0:00:00,14945598.0,14806825.0,1100.0,-791.0,1403.0
4,02-05-2026,16:21:44,0:00:00,14949143.0,14810645.0,1100.0,-791.0,1403.0


In [8]:
df.dtypes

date            str
time            str
duration        str
red         float64
ied         float64
accX        float64
accY        float64
accZ        float64
dtype: object

In [4]:
y = -1. * df['ied'].values
ymin, ymax = np.nanmin(y), np.nanmax(y)

@interact(
    start=IntSlider(min=0, max=len(y)-200, step=30, value=10000, description='start idx'),
    window=IntSlider(min=200, max=7900000, step=200, value=1000, description='window')
)
def plot_zoom(start=0, window=500):
    end = min(start + window, len(y))
    t = np.arange(start, end) / 100.0
    plt.rcParams['font.size'] = 18
    fig, ax = plt.subplots(figsize=(18, 7))
    ax.plot(t, y[start:end], 'o-', linewidth=1, markersize=2)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('IED (high-pass)')
    # ax.set_ylim([ymin, ymax])

    ax.set_title(f'Sample {start}~{end} ({window} samples / {window/100:.1f}s)')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

interactive(children=(IntSlider(value=10000, description='start idx', max=7893600, step=30), IntSlider(value=1…

In [12]:
ied_piece = y[:20000]
np.save("ied_piece.npy", ied_piece)

In [15]:
df = pd.read_csv("may_filtered.csv")

In [68]:
import pandas as pd

df = pd.read_csv("may_filtered.csv")
datetimestr = df['date'] + ' ' + df['time']
df['datetime'] = pd.to_datetime(datetimestr, format='%d-%m-%Y %H:%M:%S').astype("datetime64[s]")
df.drop(['date', 'time'], inplace=True, axis=1)

In [90]:
time = pd.unique(df.datetime).astype('int64')

,duration,red,ied,accX,accY,accZ,datetime
0,0:00:00,14936049.0,14798220.0,1100.0,-791.0,1403.0,2026-05-02 16:21:44
1,0:00:00,14938572.0,14801393.0,1100.0,-791.0,1403.0,2026-05-02 16:21:44
2,0:00:00,14941545.0,14804653.0,1100.0,-791.0,1403.0,2026-05-02 16:21:44
3,0:00:00,14945598.0,14806825.0,1100.0,-791.0,1403.0,2026-05-02 16:21:44
4,0:00:00,14949143.0,14810645.0,1100.0,-791.0,1403.0,2026-05-02 16:21:44
...,...,...,...,...,...,...,...
7893795,6:23:05,14819163.0,14292037.0,-1166.0,-171.0,1783.0,2026-05-03 18:40:54
7893796,6:23:05,14819104.0,14291981.0,-1185.0,-161.0,1783.0,2026-05-03 18:40:54
7893797,6:23:05,14819170.0,14291877.0,-1194.0,-166.0,1774.0,2026-05-03 18:40:54
7893798,6:23:05,14819193.0,14292149.0,-1188.0,-161.0,1783.0,2026-05-03 18:40:54


In [69]:
# 筛选出分组大小大于等于50的行（相当于删除了小于50的）
# 假设你用的是这种方法筛选，在末尾加上 .copy()
df_filtered = df[df.groupby('datetime')['datetime'].transform('size') >= 50].copy()

In [70]:
df_filtered['motion'] = np.linalg.norm(df_filtered[['accX', 'accY', 'accZ']].values, axis=1)

In [71]:
subdf = df_filtered[["datetime", 'ied', 'motion']].copy()

In [72]:
subdf

,datetime,ied,motion
20,2026-05-02 16:21:45,14826361.0,1950.407650
21,2026-05-02 16:21:45,14826856.0,1950.407650
22,2026-05-02 16:21:45,14827384.0,1950.407650
23,2026-05-02 16:21:45,14827099.0,1950.407650
24,2026-05-02 16:21:45,14826496.0,1950.407650
...,...,...,...
7893795,2026-05-03 18:40:54,14292037.0,2137.261332
7893796,2026-05-03 18:40:54,14291981.0,2146.912900
7893797,2026-05-03 18:40:54,14291877.0,2144.823536
7893798,2026-05-03 18:40:54,14292149.0,2148.570222


In [79]:
time = pd.unique(subdf.datetime).astype('int64')
di = np.diff(time)
gap_mask = di > 1
np.where(gap_mask)[0]

In [85]:
total_missing_time = (di[gap_mask] - 1).sum()

print(f"总共缺失了 {total_missing_time} 秒的数据")

总共缺失了 21778 秒的数据


In [93]:
df = pd.read_csv("may_filtered.csv")
df

,date,time,duration,red,ied,accX,accY,accZ
0,02-05-2026,16:21:44,0:00:00,14936049.0,14798220.0,1100.0,-791.0,1403.0
1,02-05-2026,16:21:44,0:00:00,14938572.0,14801393.0,1100.0,-791.0,1403.0
2,02-05-2026,16:21:44,0:00:00,14941545.0,14804653.0,1100.0,-791.0,1403.0
3,02-05-2026,16:21:44,0:00:00,14945598.0,14806825.0,1100.0,-791.0,1403.0
4,02-05-2026,16:21:44,0:00:00,14949143.0,14810645.0,1100.0,-791.0,1403.0
...,...,...,...,...,...,...,...,...
7893795,03-05-2026,18:40:54,6:23:05,14819163.0,14292037.0,-1166.0,-171.0,1783.0
7893796,03-05-2026,18:40:54,6:23:05,14819104.0,14291981.0,-1185.0,-161.0,1783.0
7893797,03-05-2026,18:40:54,6:23:05,14819170.0,14291877.0,-1194.0,-166.0,1774.0
7893798,03-05-2026,18:40:54,6:23:05,14819193.0,14292149.0,-1188.0,-161.0,1783.0


In [95]:
df['motion'] = np.linalg.norm(df[['accX', 'accY', 'accZ']].values, axis=1)

In [97]:
datetimestr = df['date'] + ' ' + df['time']
df['datetime'] = pd.to_datetime(datetimestr, format='%d-%m-%Y %H:%M:%S').astype("datetime64[s]")
df.drop(['date', 'time'], inplace=True, axis=1)

In [100]:
df[["datetime", 'ied', 'motion']].iloc[:80000, :].to_csv("may_piece.csv", index=False)